# 01 – Thống kê mô tả dữ liệu (W1-07) và kiểm tra chất lượng dữ liệu (W1-08)

Hướng dẫn từng bước: `docs/huong_dan/W1-07.md`, `docs/huong_dan/W1-08.md`.
Ô có chữ **TODO** là phần m tự viết. Các ô còn lại t viết sẵn, m chạy lần lượt từ trên xuống.
Hình vẽ lưu vào `results/figures/eda_*.png` (nằm trong Drive chung) để dán vào báo cáo mục 2.

In [ ]:
# Ô 0 – cài đặt môi trường (mỗi lần mở notebook đều chạy ô này trước; Colab hỏi quyền truy cập Drive thì bấm cho phép)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NLI'      # lối tắt tới Drive chung (xem docs/huong_dan/README.md, mục đầu)
!git clone -q https://github.com/quanuiter/NLI.git /content/NLI 2>/dev/null || git -C /content/NLI pull -q
%cd /content/NLI
!pip install -q -r requirements.txt
import os
for sub in ['data/processed', 'tools', 'results']:
    target = os.path.join(DRIVE_DIR, sub); os.makedirs(target, exist_ok=True)
    if os.path.islink(sub): continue
    if os.path.isdir(sub):
        !cp -rn {sub}/. {target}/ && rm -rf {sub}
    os.symlink(target, sub)

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from src.data import DATASET_SPLITS, LABELS, load_split
from src.embeddings import tokenize

# Gộp toàn bộ dữ liệu vào bảng df (phải có 7 tập, tổng 119.776 dòng)
df = pd.concat([load_split(d, s) for d, splits in DATASET_SPLITS.items() for s in splits], ignore_index=True)
df['tap'] = df['source'] + '_' + df['split']
df.groupby('tap').size()

## 1. Phân bố nhãn

In [ ]:
pd.crosstab(df['tap'], df['label']).reindex(columns=LABELS)

**TODO 1:** đổi bảng trên sang tỉ lệ % (gợi ý: thêm `normalize='index'` vào `pd.crosstab(...)`), rồi viết 1–2 câu nhận xét:
bộ nào cân bằng nhãn, bộ nào lệch về nhãn nào.

In [ ]:
# TODO 1

## 2. Độ dài câu (số âm tiết)

In [ ]:
df['len_p'] = df['premise'].map(lambda s: len(tokenize(s)))
df['len_h'] = df['hypothesis'].map(lambda s: len(tokenize(s)))
df.groupby('tap')[['len_p', 'len_h']].agg(['mean', 'median', 'max']).round(1)

**TODO 2:** vẽ histogram độ dài hypothesis của 3 tập `xnli_train`, `vianli_train`, `vinli_test` trên cùng một hình,
lưu `results/figures/eda_do_dai.png`. Code mẫu có trong `docs/huong_dan/W1-07.md`.
Nhận xét: bộ nào có câu dài hơn? Độ dài hypothesis có khác nhau giữa các nhãn không? (xem bằng `df.groupby(['tap','label'])['len_h'].mean()`)

In [ ]:
# TODO 2

## 3. Độ trùng lặp từ giữa premise và hypothesis (Jaccard, %)

In [ ]:
def jaccard(p, h):
    a, b = set(t.lower() for t in tokenize(p)), set(t.lower() for t in tokenize(h))
    return len(a & b) / max(1, len(a | b))

df['jaccard'] = [jaccard(p, h) for p, h in zip(df['premise'], df['hypothesis'])]
(df.pivot_table(index='label', columns='tap', values='jaccard') * 100).round(1).reindex(LABELS)

**TODO 3:** vẽ biểu đồ cột cho bảng trên (gợi ý: `bang.plot.bar()` rồi `plt.savefig('results/figures/eda_jaccard.png', dpi=150, bbox_inches='tight')`).
Nhận xét: ở bộ nào các cặp entailment có độ trùng lặp từ cao nhất? ViANLI có ngược lại không? (Số liệu phải khớp `docs/ket_qua_m2.md` mục 4.)

In [ ]:
# TODO 3

## 4. Tần suất từ phủ định theo nhãn

In [ ]:
NEG = r'\b(không|chưa|chẳng|chả|đâu có|không hề|chưa từng)\b'
# TODO 4: 1) tạo cột đánh dấu hypothesis có từ phủ định:  df['h_neg'] = df['hypothesis'].str.contains(NEG, case=False)
#         2) tính tỉ lệ theo nhãn và theo tập:        df.pivot_table(index='label', columns='tap', values='h_neg')
#         Câu hỏi: ở XNLI, từ phủ định xuất hiện nhiều nhất ở nhãn nào? ViNLI, ViANLI có giống không?
#         (bài ViANLI mục 7.6: ở ViNLI, từ "không" lại xuất hiện nhiều ở nhãn entailment)

---
# Phần W1-08: kiểm tra chất lượng dữ liệu

In [ ]:
# 5. Câu rỗng hoặc quá ngắn (câu như "Đúng vậy." vẫn hợp lệ, không tính là lỗi)
df[(df['len_p'] < 3) | (df['len_h'] < 2)][['tap', 'premise', 'hypothesis', 'label']].head(20)

In [ ]:
# 6. Mẫu trùng lặp trong cùng một tập
dup = df[df.duplicated(['tap', 'premise', 'hypothesis'], keep=False)]
print(dup.groupby('tap').size())
# TODO 6: trong các mẫu trùng lặp, có cặp nào giống hệt nhau nhưng KHÁC nhãn không? (code mẫu trong docs/huong_dan/W1-08.md)

In [ ]:
# 7. Rò rỉ dữ liệu: cặp câu của tập test có xuất hiện trong tập train không?
def leak(train_tap, test_tap):
    tr = set(zip(df.loc[df.tap == train_tap, 'premise'], df.loc[df.tap == train_tap, 'hypothesis']))
    te = df[df.tap == test_tap]
    n = sum((p, h) in tr for p, h in zip(te['premise'], te['hypothesis']))
    return n, len(te)

for a, b in [('xnli_train', 'xnli_test'), ('vianli_train', 'vianli_test'), ('vianli_train', 'vinli_test')]:
    print(a, '->', b, leak(a, b))
# TODO 7: làm tương tự nhưng chỉ so sánh premise, ghi lại số lượng

**TODO 8 – bảng tổng kết:** *loại lỗi · số lượng · cách xử lý*. Các lỗi t đã xử lý có sẵn trong `docs/huong_dan/W1-08.md`,
m chép sang rồi bổ sung kết quả ô 5–7. Bảng này đưa vào báo cáo mục 2.